# Add Proteins to Merged Datasets

Adds protein MB-selected features to each of the 8 existing Clinical+RNA+CNV+MUT merged files.

**Run cells in order. Restart kernel before running if proteins were already added.**

**Input:** `MERGE/01_ultra_conservative.csv` ... `08_composite.csv`  
**Output:** same files updated with `PROT_` columns + `merge_summary_5modalities.csv`

## Setup — Paths, Dataset Pairs & Best Proteins Config

In [3]:
"""
ADD PROTEINS: Add protein features to existing Clinical+RNA+CNV+MUT merged datasets.

Script location: 01_Causal_feature_extraction/
Input/Output:    01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

PROT_DIR   = SCRIPT_DIR / "Proteins"
FILT_DIR   = PROT_DIR / "statistical_filtered"
MB_DIR     = PROT_DIR / "mb_results"
MERGE_DIR  = SCRIPT_DIR / "MERGE"

print(f"Script dir:   {SCRIPT_DIR}")
print(f"Proteins dir: {PROT_DIR.exists()} -> {PROT_DIR}")
print(f"Proteins MB:  {MB_DIR.exists()} -> {MB_DIR}")
print(f"MERGE dir:    {MERGE_DIR.exists()}")

DATASET_PAIRS = {
    "01_ultra_conservative": "prot_1_ultra_conservative_50proteins",
    "02_conservative":       "prot_2_conservative_50proteins",
    "03_standard":           "prot_3_standard_50proteins",
    "04_fdr_significant":    "prot_4_fdr_significant_200proteins",
    "05_balanced":           "prot_5_balanced_50proteins",
    "06_correlation":        "prot_6_correlation_50proteins",
    "07_top_correlated":     "prot_7_top_correlated_200proteins",
    "08_composite":          "prot_8_composite_200proteins",
}

# ---------------------------------------------------------------------------
# STEP 1: LOAD PROTEINS SUMMARY AND SELECT BEST CONFIG PER DATASET
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 1: SELECT BEST PROTEINS CONFIG PER DATASET (highest C-index)")
print("="*70)
print(f"Reading: {MB_DIR / 'summary_all_results.csv'}")

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")
print(f"Summary shape: {summary.shape}")
print(f"Datasets in summary: {summary['dataset'].unique().tolist()}")

best_prot = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index"]]

print(best_prot.to_string(index=False))

Script dir:   C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
Proteins dir: True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins
Proteins MB:  True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins\mb_results
MERGE dir:    True

STEP 1: SELECT BEST PROTEINS CONFIG PER DATASET (highest C-index)
Reading: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins\mb_results\summary_all_results.csv
Summary shape: (56, 15)
Datasets in summary: ['prot_1_ultra_conservative_50proteins', 'prot_2_conservative_50proteins', 'prot_3_standard_50proteins', 'prot_4_fdr_significant_200proteins', 'prot_5_balanced_50proteins', 'prot_6_correlation_50proteins', 'prot_7_top_correlated_200proteins', 'prot_8_composite_200proteins']
                             dataset algorithm  alpha  c_index
   prot_7_top_correlated_200proteins      MMMB   0.05 0.606519
          prot_3_standard_50proteins      IAMB   0.05 0.488617
      prot_2_cons

## Helper Functions

In [5]:
# ---------------------------------------------------------------------------
# STEP 2: HELPERS
# ---------------------------------------------------------------------------
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = MB_DIR / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found: {MB_DIR / dataset / f'{algorithm}_alpha{alpha:.2f}_genes.txt'}"
    )


def load_proteins(prot_dataset, algorithm, alpha):
    candidates = list(FILT_DIR.glob(f"{prot_dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No proteins file found for '{prot_dataset}'")
    prot_file = candidates[0]

    genes      = load_genes(prot_dataset, algorithm, alpha)
    prot       = pd.read_csv(prot_file, index_col=0)
    available  = [g for g in genes if g in prot.columns]
    missing_g  = [g for g in genes if g not in prot.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} proteins not in file")
    if not available:
        raise ValueError(f"No proteins available in {prot_file.name}")
    prot = prot[available].copy()
    print(f"  Proteins loaded: {prot.shape}  ({prot_file.name})")

    tumor_mask = prot.index.str.contains(r"-01[A-Z]?$", regex=True)
    prot = prot[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(prot)} samples")

    prot.index = [normalize_id(i) for i in prot.index]

    if prot.index.duplicated().any():
        n_pts = prot.index[prot.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate samples")
        prot = prot.groupby(prot.index).mean()

    prot.columns = [f"PROT_{c}" for c in prot.columns]
    return prot

print("\nHelper functions ready")


Helper functions ready


## Add Proteins to All 8 Datasets

In [7]:
# ---------------------------------------------------------------------------
# STEP 3: ADD PROTEINS TO ALL 8 DATASETS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: ADD PROTEINS TO ALL 8 MERGED DATASETS")
print("="*70)

results = []

for short_name, prot_dataset in DATASET_PAIRS.items():
    merged_file = MERGE_DIR / f"{short_name}.csv"
    if not merged_file.exists():
        print(f"\nSkipping {short_name} — file not found")
        continue

    row = best_prot[best_prot["dataset"] == prot_dataset]
    if row.empty:
        print(f"\nNo proteins summary entry for {prot_dataset} — skipping")
        continue
    row = row.iloc[0]

    print(f"\n{'─'*70}")
    print(f"[{short_name}]")
    print(f"  Proteins dataset: {prot_dataset}")
    print(f"  Config: {row['algorithm']}  alpha={row['alpha']}  C-index={row['c_index']:.4f}")

    try:
        merged = pd.read_csv(merged_file, index_col=0)

        # drop any PROT_ columns from a previous run
        prot_cols_existing = [c for c in merged.columns if c.startswith("PROT_")]
        if prot_cols_existing:
            merged = merged.drop(columns=prot_cols_existing)
            print(f"  Dropped {len(prot_cols_existing)} existing PROT_ columns")

        n_clin = sum(1 for c in merged.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in merged.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in merged.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in merged.columns if c.startswith("MUT_"))
        print(f"  Existing: {merged.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  MUT_={n_mut}")

        prot = load_proteins(prot_dataset, row["algorithm"], float(row["alpha"]))

        common = sorted(set(merged.index) & set(prot.index))
        print(f"  Common patients: {len(common)}  "
              f"(merged={len(merged)}, proteins={len(prot)})")
        if not common:
            raise ValueError("No common patients")

        final = pd.concat([merged.loc[common], prot.loc[common]], axis=1)

        assert final.isna().sum().sum() == 0, \
            f"Missing values: {final.isna().sum().sum()}"

        n_prot = sum(1 for c in final.columns if c.startswith("PROT_"))
        n_clin = sum(1 for c in final.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in final.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in final.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in final.columns if c.startswith("MUT_"))
        print(f"  Final: {final.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  "
              f"MUT_={n_mut}  PROT_={n_prot}  | no missing")

        final.to_csv(merged_file)
        print(f"  Saved: {merged_file.name}")

        results.append({
            "file":             short_name + ".csv",
            "n_samples":        final.shape[0],
            "n_clin":           n_clin,
            "n_rna":            n_rna,
            "n_cnv":            n_cnv,
            "n_mut":            n_mut,
            "n_prot":           n_prot,
            "n_total":          final.shape[1],
            "prot_dataset":     prot_dataset,
            "prot_algorithm":   row["algorithm"],
            "prot_alpha":       row["alpha"],
            "prot_c_index":     row["c_index"],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


STEP 3: ADD PROTEINS TO ALL 8 MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  Proteins dataset: prot_1_ultra_conservative_50proteins
  Config: GSMB  alpha=0.05  C-index=0.4693
  Existing: (923, 294)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  Proteins loaded: (859, 50)  (prot_1_ultra_conservative_50proteins.csv)
  After Primary Tumor filter: 858 samples
  Common patients: 745  (merged=923, proteins=858)
  Final: (745, 344)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50  PROT_=50  | no missing
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  Proteins dataset: prot_2_conservative_50proteins
  Config: GSMB  alpha=0.2  C-index=0.4789
  Existing: (923, 294)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  Proteins loaded: (859, 50)  (prot_2_conservative_50proteins.csv)
  After Primary Tumor filter: 858 samples
  Common patients: 745  (merged=923, proteins=858)
  Final: 

## Summary

In [9]:
# ---------------------------------------------------------------------------
# STEP 4: SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(MERGE_DIR / "merge_summary_5modalities.csv", index=False)
    print(f"\nCLIN_ same:  {df['n_clin'].nunique()==1}  ({df['n_clin'].iloc[0]})")
    print(f"RNA_ varies: {df['n_rna'].nunique()>1}  ({df['n_rna'].min()}–{df['n_rna'].max()})")
    print(f"CNV_ varies: {df['n_cnv'].nunique()>1}  ({df['n_cnv'].min()}–{df['n_cnv'].max()})")
    print(f"MUT_ varies: {df['n_mut'].nunique()>1}  ({df['n_mut'].min()}–{df['n_mut'].max()})")
    print(f"PROT_ varies:{df['n_prot'].nunique()>1}  ({df['n_prot'].min()}–{df['n_prot'].max()})")
    print(f"Total:        {df['n_total'].min()}–{df['n_total'].max()} features")
    print(f"Samples:      {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nSaved: {MERGE_DIR / 'merge_summary_5modalities.csv'}")
else:
    print("No datasets processed successfully.")


STEP 4: SUMMARY
                     file  n_samples  n_clin  n_rna  n_cnv  n_mut  n_prot  n_total                         prot_dataset prot_algorithm  prot_alpha  prot_c_index
01_ultra_conservative.csv        745     144     50     50     50      50      344 prot_1_ultra_conservative_50proteins           GSMB        0.05      0.469252
      02_conservative.csv        745     144     50     50     50      50      344       prot_2_conservative_50proteins           GSMB        0.20      0.478856
          03_standard.csv        745     144     50     50     50      50      344           prot_3_standard_50proteins           IAMB        0.05      0.488617
   04_fdr_significant.csv        745     144     50     50     50      50      344   prot_4_fdr_significant_200proteins           MMMB        0.05      0.412913
          05_balanced.csv        745     144     50     50     50      50      344           prot_5_balanced_50proteins           IAMB        0.05      0.411268
       06_correla